# Pain EEG pipeline (Colab)

The whole pipeline is in this notebook. Part 1 does the preprocessing and connectivity. Part 2 is the
Modares-Haghighi graph-metric + SVM baseline. Each code cell below writes one module to
`/content/pipeline`. **Edit the code here**, then rerun that cell.

The only thing you need on Google Drive is the EEG data: a folder with the `.gdf` files and
`manifest.csv` (e.g. `MyDrive/generic_dataset`). Features and results are saved to Drive.

Run the cells top to bottom (or *Runtime → Run all*).

## 0. Install dependencies
`torch` is already on Colab. `torch_geometric` is only needed later for the GAT.

In [ ]:
!pip install -q "mne>=1.8" "mne-icalabel>=0.7" onnxruntime mlxtend bctpy
!mkdir -p /content/pipeline/paineeg && touch /content/pipeline/paineeg/__init__.py
import os; print('CPU cores:', os.cpu_count())

## 1. Code: Part 1 (shared preprocessing)
### `paineeg/config.py`: every setting in one place

In [ ]:
%%writefile /content/pipeline/paineeg/config.py
"""Shared settings for the whole pipeline.

Both models (the Modares-Haghighi baseline and the GAT) read the SAME
preprocessed data and the SAME connectivity matrices, so every setting
in this file applies to both. Change values here, not inside the scripts.
"""
from dataclasses import dataclass, field
from typing import Optional, Sequence


@dataclass
class Config:
    # ---------------- Frequency bands ----------------
    # Theta is the main band for dysmenorrhea (Lee et al., 2017).
    # Alpha is the band used by Modares-Haghighi et al. (2021).
    bands: dict = field(default_factory=lambda: {
        "theta": (4.0, 8.0),
        "alpha": (8.0, 12.0),
    })

    # ---------------- Channels ----------------
    # Fix the channel list so every subject's graph has the same nodes in the
    # same order. None = use all EEG channels in the first file (all files
    # must then have the same channels).
    channels: Optional[Sequence[str]] = None
    montage: str = "standard_1020"

    # ---------------- Preprocessing ----------------
    l_freq: float = 0.5           # Modares-Haghighi: 0.5-45 Hz band-pass
    h_freq: float = 45.0
    line_freq: Optional[float] = 50.0   # 50 Hz (Australia/Europe/Iran), 60 Hz (USA)
    resample_sfreq: Optional[float] = 250.0
    crop: Optional[tuple] = None  # (tmin, tmax) in seconds, e.g. pick the eyes-closed segment
    bad_channel_z: float = 5.0    # flag channels whose robust log-variance z-score exceeds this (PREP-style)
    use_ica: bool = True          # ICA + ICLabel (replaces the paper's SOBI step)
    ica_n_components: Optional[int] = None  # None = min(n_channels - 1, 20)
    ica_l_freq: float = 1.0       # ICA/ICLabel is fitted on a 1-100 Hz copy (what ICLabel was trained on);
    ica_h_freq: float = 100.0     # the components are then removed from the 0.5-45 Hz data
    iclabel_exclude: tuple = ("eye blink", "muscle artifact", "heart beat",
                              "line noise", "channel noise")
    iclabel_threshold: float = 0.7

    # ---------------- Windows ----------------
    win_sec: float = 1.0          # Modares-Haghighi: 1 s windows ...
    overlap: float = 0.5          # ... with 50 % overlap
    reject_uv: float = 100.0      # drop windows with any sample above +/-100 uV

    # ---------------- Connectivity ----------------
    mvar_order: object = "bic"    # "bic" or an integer model order
    mvar_max_order: int = 10
    mvar_ridge: float = 1e-6      # small ridge term keeps the MVAR fit stable on 1 s windows

    # ---------------- Graph construction ----------------
    density: float = 0.9          # keep strongest 90 % of edges (best density in the paper)

    random_seed: int = 42


DEFAULT = Config()

### `paineeg/preprocess.py`: load → filter → ICA + ICLabel → windows

In [ ]:
%%writefile /content/pipeline/paineeg/preprocess.py
"""Step 1 - EEG preprocessing (shared by both models).

Follows Modares-Haghighi et al. (2021) section 3.1, adapted to MNE-Python:
  * band-pass 0.5-45 Hz + notch at the line frequency
  * artifact removal: the paper used SOBI + fractal dimension (EOG) and
    spectral similarity to Fp1 (EMG). MNE has no SOBI, so we use ICA with
    ICLabel, which removes the same artifact classes automatically.
Added for robustness: bad-channel detection/interpolation and average reference.
"""
import warnings

import numpy as np
import mne

warnings.filterwarnings("ignore", category=RuntimeWarning, module="mne_icalabel")

from .config import Config


def make_montage(name):
    """standard_1020 is renamed colin27_1020 in newer MNE; use whichever exists."""
    builtin = mne.channels.get_builtin_montages()
    if name == "standard_1020" and "colin27_1020" in builtin:
        name = "colin27_1020"
    return mne.channels.make_standard_montage(name)


def load_raw(path):
    """Read any MNE-supported EEG file (.fif, .edf, .bdf, .gdf, .set, .vhdr ...)."""
    raw = mne.io.read_raw(str(path), preload=True, verbose="error")
    return raw


def _pick_channels(raw, channels):
    """Keep only `channels`, in that order, matching names case-insensitively."""
    lookup = {ch.lower().replace(" ", ""): ch for ch in raw.ch_names}
    rename, missing = {}, []
    for want in channels:
        key = want.lower().replace(" ", "")
        if key in lookup:
            rename[lookup[key]] = want
        else:
            missing.append(want)
    if missing:
        raise ValueError(f"Channels missing from recording: {missing}. "
                         f"Available: {raw.ch_names}")
    raw.rename_channels(rename)
    raw.set_channel_types({ch: "eeg" for ch in channels}, verbose="error")
    raw.pick(list(channels))
    raw.reorder_channels(list(channels))
    return raw


def _find_bad_channels(raw, z_thresh):
    """Flag flat channels and channels with extreme variance (robust z-score)."""
    data = raw.get_data()
    var = np.var(data, axis=1)
    bads = [ch for ch, v in zip(raw.ch_names, var) if v < 1e-20]
    logv = np.log(var + 1e-30)
    med = np.median(logv)
    mad = np.median(np.abs(logv - med)) * 1.4826 + 1e-12
    z = (logv - med) / mad
    bads += [ch for ch, zz in zip(raw.ch_names, z) if abs(zz) > z_thresh and ch not in bads]
    return bads


def _run_ica(raw, raw_fit, cfg: Config):
    """ICA + ICLabel. Fits on `raw_fit` (1-100 Hz, as ICLabel expects) and removes
    the artifact components from `raw`. Returns the cleaned raw and the excluded labels."""
    from mne_icalabel import label_components

    n_good = len(raw.ch_names) - len(raw.info["bads"])
    n_comp = cfg.ica_n_components or min(n_good - 1, 20)
    ica = mne.preprocessing.ICA(n_components=n_comp, method="infomax",
                                fit_params=dict(extended=True),
                                random_state=cfg.random_seed, max_iter="auto",
                                verbose="error")
    ica.fit(raw_fit, verbose="error")
    labels = label_components(raw_fit, ica, method="iclabel")
    exclude = [i for i, (lab, p) in enumerate(zip(labels["labels"], labels["y_pred_proba"]))
               if lab in cfg.iclabel_exclude and p >= cfg.iclabel_threshold]
    ica.exclude = exclude
    ica.apply(raw, verbose="error")
    return raw, [labels["labels"][i] for i in exclude]


def preprocess(raw, cfg: Config):
    """Run the full cleaning chain. Returns (clean_raw, log_dict)."""
    raw = raw.copy()
    log = {}
    if cfg.channels:
        raw = _pick_channels(raw, cfg.channels)
    else:
        raw.pick("eeg")
    if cfg.crop:
        raw.crop(*cfg.crop)

    montage = make_montage(cfg.montage)
    raw.set_montage(montage, match_case=False, on_missing="warn", verbose="error")

    if cfg.resample_sfreq and raw.info["sfreq"] > cfg.resample_sfreq:
        raw.resample(cfg.resample_sfreq, verbose="error")

    nyq = raw.info["sfreq"] / 2
    if cfg.line_freq and cfg.line_freq < nyq:
        raw.notch_filter(np.arange(cfg.line_freq, nyq, cfg.line_freq), verbose="error")
    # ICLabel was trained on 1-100 Hz data, so ICA is fitted on a separate copy
    # taken before the analysis band-pass
    if cfg.use_ica:
        raw_fit = raw.copy().filter(cfg.ica_l_freq, min(cfg.ica_h_freq, nyq - 1), verbose="error")
    raw.filter(cfg.l_freq, min(cfg.h_freq, nyq - 1), verbose="error")

    raw.info["bads"] = _find_bad_channels(raw, cfg.bad_channel_z)
    log["bad_channels"] = list(raw.info["bads"])

    raw.set_eeg_reference("average", projection=False, verbose="error")

    if cfg.use_ica:
        raw_fit.info["bads"] = list(raw.info["bads"])
        raw_fit.set_eeg_reference("average", projection=False, verbose="error")
        raw, excluded = _run_ica(raw, raw_fit, cfg)
        log["ica_excluded"] = excluded
    if raw.info["bads"]:
        raw.interpolate_bads(reset_bads=True, verbose="error")
        raw.set_eeg_reference("average", projection=False, verbose="error")
    return raw, log


def window_starts(n_times, sfreq, cfg: Config):
    """Sample indices where each window starts (1 s windows, 50 % overlap by default)."""
    win = int(round(cfg.win_sec * sfreq))
    step = max(1, int(round(win * (1 - cfg.overlap))))
    return np.arange(0, n_times - win + 1, step), win


def make_windows(raw, cfg: Config):
    """Cut the clean recording into windows and drop windows above +/- reject_uv.

    Returns data (n_win, n_ch, n_samples), start times (s), and the kept start
    indices (needed to slice band-filtered signals for wPLI).
    """
    data = raw.get_data(reject_by_annotation="NaN")
    sfreq = raw.info["sfreq"]
    starts, win = window_starts(data.shape[1], sfreq, cfg)
    thr = cfg.reject_uv * 1e-6
    keep = []
    for s in starts:
        seg = data[:, s:s + win]
        if np.isnan(seg).any():
            continue
        if np.abs(seg).max() > thr:
            continue
        keep.append(s)
    keep = np.array(keep, dtype=int)
    if keep.size == 0:
        return np.empty((0, data.shape[0], win)), keep / sfreq, keep
    windows = np.stack([data[:, s:s + win] for s in keep])
    return windows, keep / sfreq, keep

### `paineeg/connectivity.py`: PDC, wPLI, band power

In [ ]:
%%writefile /content/pipeline/paineeg/connectivity.py
"""Step 2 - connectivity per window (shared by both models).

PDC (partial directed coherence) - the measure used by Modares-Haghighi et al.
  An MVAR model is fitted to each 1 s window (order chosen by BIC, as in the
  paper). PDC is computed from the MVAR coefficients and averaged over the
  frequencies of each band. Deviation from the paper: the MVAR is fitted on the
  broadband (0.5-45 Hz) window and PDC is read out per band, because fitting an
  MVAR to an already narrow-band-filtered signal is numerically ill-conditioned.

wPLI (weighted phase lag index) - recommended alternative (robust to volume
  conduction). Computed from the analytic (Hilbert) signal of each band, over
  the samples inside each window.

Matrix convention for BOTH measures: W[source, target]
(row = sender, column = receiver). wPLI is symmetric, PDC is directed.
"""
import numpy as np
from scipy.signal import hilbert, welch

import mne


# ------------------------------------------------------------------ MVAR / PDC
def fit_mvar(x, order, ridge=1e-6):
    """Least-squares MVAR fit. x: (n_ch, n_times). Returns lags (p, C, C), noise cov."""
    x = x - x.mean(axis=1, keepdims=True)
    C, T = x.shape
    Y = x[:, order:]
    Z = np.vstack([x[:, order - k:T - k] for k in range(1, order + 1)])  # (C*p, T-p)
    ZZ = Z @ Z.T
    ZZ += ridge * np.trace(ZZ) / ZZ.shape[0] * np.eye(ZZ.shape[0])
    A = np.linalg.solve(ZZ, Z @ Y.T).T          # (C, C*p)
    E = Y - A @ Z
    sigma = E @ E.T / E.shape[1]
    lags = np.stack([A[:, k * C:(k + 1) * C] for k in range(order)])
    return lags, sigma


def bic_order(x, max_order, ridge=1e-6):
    """Model order that minimises the Bayesian information criterion (Schwarz, 1978)."""
    C, T = x.shape
    best, best_bic = 1, np.inf
    for p in range(1, max_order + 1):
        n = T - p
        if n <= C * p:                      # not enough samples for this order
            break
        _, sigma = fit_mvar(x, p, ridge)
        sign, logdet = np.linalg.slogdet(sigma)
        if sign <= 0:
            continue
        bic = logdet + np.log(n) * p * C * C / n
        if bic < best_bic:
            best, best_bic = p, bic
    return best


def pdc_from_lags(lags, freqs, sfreq):
    """PDC spectrum (Baccala & Sameshima, 2001). Returns P[f, i, j] = flow j -> i."""
    p, C, _ = lags.shape
    k = np.arange(1, p + 1)
    E = np.exp(-2j * np.pi * np.outer(freqs, k) / sfreq)          # (F, p)
    Af = np.eye(C)[None] - np.einsum("fk,kij->fij", E, lags)       # (F, C, C)
    denom = np.sqrt((np.abs(Af) ** 2).sum(axis=1, keepdims=True))  # column norms
    return np.abs(Af) / denom


def choose_subject_order(windows, cfg, n_probe=30):
    """One MVAR order per subject: median BIC order over a sample of windows."""
    if isinstance(cfg.mvar_order, int):
        return cfg.mvar_order
    idx = np.linspace(0, len(windows) - 1, min(n_probe, len(windows))).astype(int)
    orders = [bic_order(windows[i], cfg.mvar_max_order, cfg.mvar_ridge) for i in idx]
    return int(np.median(orders))


def pdc_windows(windows, sfreq, bands, order, ridge=1e-6, freq_step=0.5):
    """PDC per window and band. Returns (n_win, n_bands, C, C) with W[source, target]."""
    n_win, C, _ = windows.shape
    out = np.zeros((n_win, len(bands), C, C))
    band_freqs = [np.arange(lo, hi + 1e-9, freq_step) for lo, hi in bands.values()]
    for w in range(n_win):
        lags, _ = fit_mvar(windows[w], order, ridge)
        for b, freqs in enumerate(band_freqs):
            P = pdc_from_lags(lags, freqs, sfreq).mean(axis=0)  # P[i, j] = j -> i
            W = P.T.copy()                                         # W[source, target]
            np.fill_diagonal(W, 0.0)
            out[w, b] = W
    return out


# ------------------------------------------------------------------ wPLI
def band_analytic(raw, bands):
    """Band-filter the continuous clean recording and return analytic signals.

    Filtering the continuous signal (not each 1 s window) avoids edge artefacts.
    Returns array (n_bands, C, n_times), complex.
    """
    data = raw.get_data()
    sfreq = raw.info["sfreq"]
    out = []
    for lo, hi in bands.values():
        filt = mne.filter.filter_data(data, sfreq, lo, hi, verbose="error")
        out.append(hilbert(filt, axis=-1))
    return np.stack(out)


def wpli_windows(analytic, starts, win):
    """Time-resolved wPLI (Vinck et al., 2011) within each window.

    wPLI_ij = |mean_t Im(z_i z_j*)| / mean_t |Im(z_i z_j*)|
    Returns (n_win, n_bands, C, C), symmetric, zero diagonal.
    """
    n_b, C, _ = analytic.shape
    out = np.zeros((len(starts), n_b, C, C))
    for w, s in enumerate(starts):
        z = analytic[:, :, s:s + win]                         # (B, C, T)
        cross = np.einsum("bit,bjt->bijt", z, np.conj(z)).imag
        num = np.abs(cross.mean(axis=-1))
        den = np.abs(cross).mean(axis=-1) + 1e-20
        W = num / den
        for b in range(n_b):
            np.fill_diagonal(W[b], 0.0)
        out[w] = W
    return out


# ------------------------------------------------------------------ band power
def band_power_windows(windows, sfreq, bands):
    """log10 band power per window, band and channel -> (n_win, n_bands, C).

    Used as GAT node features and as model A (band-power baseline).
    """
    n_win, C, T = windows.shape
    f, psd = welch(windows, fs=sfreq, nperseg=min(T, int(sfreq)), axis=-1)
    out = np.zeros((n_win, len(bands), C))
    for b, (lo, hi) in enumerate(bands.values()):
        m = (f >= lo) & (f < hi)
        out[:, b] = np.log10(psd[..., m].mean(axis=-1) + 1e-30)
    return out

### `paineeg/data.py`: loads the saved per-subject files

In [ ]:
%%writefile /content/pipeline/paineeg/data.py
"""Load the per-subject .npz files written by 01_build_dataset.py.

Each file holds everything both models need for one participant:
  conn_pdc   (n_win, n_bands, C, C)   PDC, W[source, target]
  conn_wpli  (n_win, n_bands, C, C)   wPLI, symmetric
  bandpower  (n_win, n_bands, C)      log10 power (GAT node features)
  t_start    (n_win,)                 window start times (s)
  label, subject, ch_names, bands, sfreq, mvar_order
"""
from pathlib import Path
import numpy as np


def load_subjects(feature_dir, conn="pdc"):
    """Return a list of dicts, one per subject, with the chosen connectivity."""
    subjects = []
    for f in sorted(Path(feature_dir).glob("*.npz")):
        d = np.load(f, allow_pickle=True)
        key = f"conn_{conn}"
        if key not in d.files:
            raise KeyError(f"{f.name} has no {key}; rebuild with --conn {conn} or both")
        if d[key].shape[0] == 0:
            print(f"[skip] {f.name}: no clean windows")
            continue
        subjects.append(dict(
            subject=str(d["subject"]),
            label=int(d["label"]),
            conn=d[key].astype(np.float32),
            bandpower=d["bandpower"].astype(np.float32),
            ch_names=[str(c) for c in d["ch_names"]],
            bands=[str(b) for b in d["bands"]],
            directed=(conn == "pdc"),
        ))
    if not subjects:
        raise FileNotFoundError(f"No subject files found in {feature_dir}")
    chs = subjects[0]["ch_names"]
    for s in subjects:
        if s["ch_names"] != chs:
            raise ValueError(f"Subject {s['subject']} has different channels; "
                             "set Config.channels so every graph has the same nodes")
    return subjects

### `01_build_dataset.py`: runs Part 1 over a manifest

In [ ]:
%%writefile /content/pipeline/01_build_dataset.py
"""STEP 1 (shared): preprocess every recording and save connectivity + band power.

Both models read the output of this script, so they see identical data.

Usage
-----
python 01_build_dataset.py --manifest data/manifest.csv --out features/ \
       --conn both --label-col label

The manifest is a CSV with at least these columns:
  subject  - participant ID (used for subject-wise cross-validation)
  path     - path to the EEG file (.fif/.edf/.bdf/.gdf/.set/.vhdr ...)
  label    - integer class (e.g. 0 = control, 1 = pain), or any column named by --label-col

Optional flags:
  --snr 20      add white noise at this SNR (dB) after cleaning
                (repeats the paper's robustness test; save to a separate --out)
  --no-ica      skip ICA/ICLabel (faster; for quick tests only)
  --channels Fp1,Fp2,...   fix the node set (strongly recommended for real data)
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from paineeg.config import Config
from paineeg import preprocess as pp
from paineeg import connectivity as cx


def add_white_noise(raw, snr_db, rng):
    data = raw.get_data()
    p_sig = np.mean(data ** 2, axis=1, keepdims=True)
    p_noise = p_sig / (10 ** (snr_db / 10))
    raw._data = data + rng.standard_normal(data.shape) * np.sqrt(p_noise)
    return raw


def process_subject(row, cfg, conn, label_col, snr=None):
    raw = pp.load_raw(row["path"])
    raw, log = pp.preprocess(raw, cfg)
    if snr is not None:
        raw = add_white_noise(raw, snr, np.random.default_rng(cfg.random_seed))
    sfreq = raw.info["sfreq"]
    windows, t_start, starts = pp.make_windows(raw, cfg)
    win = windows.shape[-1]
    out = dict(subject=str(row["subject"]), label=int(row[label_col]),
               ch_names=np.array(raw.ch_names), bands=np.array(list(cfg.bands)),
               sfreq=sfreq, t_start=t_start)
    out["bandpower"] = cx.band_power_windows(windows, sfreq, cfg.bands) if len(windows) else \
        np.empty((0, len(cfg.bands), len(raw.ch_names)))
    if conn in ("pdc", "both"):
        order = cx.choose_subject_order(windows, cfg) if len(windows) else 0
        out["mvar_order"] = order
        out["conn_pdc"] = cx.pdc_windows(windows, sfreq, cfg.bands, order, cfg.mvar_ridge) \
            if len(windows) else np.empty((0, len(cfg.bands), len(raw.ch_names), len(raw.ch_names)))
    if conn in ("wpli", "both"):
        analytic = cx.band_analytic(raw, cfg.bands)
        out["conn_wpli"] = cx.wpli_windows(analytic, starts, win)
    log.update(n_windows=int(len(windows)), mvar_order=int(out.get("mvar_order", 0)))
    return out, log


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--conn", choices=["pdc", "wpli", "both"], default="both")
    ap.add_argument("--label-col", default="label")
    ap.add_argument("--channels", default=None, help="comma-separated channel list")
    ap.add_argument("--line-freq", type=float, default=None)
    ap.add_argument("--crop", default=None, help="tmin,tmax in seconds")
    ap.add_argument("--no-ica", action="store_true")
    ap.add_argument("--snr", type=float, default=None)
    args = ap.parse_args()

    cfg = Config()
    if args.channels:
        cfg.channels = [c.strip() for c in args.channels.split(",")]
    if args.line_freq is not None:
        cfg.line_freq = args.line_freq
    if args.crop:
        cfg.crop = tuple(float(v) for v in args.crop.split(","))
    if args.no_ica:
        cfg.use_ica = False

    man = pd.read_csv(args.manifest)
    base = Path(args.manifest).parent
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    logs = []
    for _, row in man.iterrows():
        row = row.copy()
        p = Path(row["path"])
        row["path"] = p if p.is_absolute() else (base / p)
        t0 = time.time()
        try:
            out, log = process_subject(row, cfg, args.conn, args.label_col, args.snr)
        except Exception as e:  # keep going; report at the end
            print(f"[FAIL] {row['subject']}: {e}")
            logs.append(dict(subject=row["subject"], error=str(e)))
            continue
        np.savez_compressed(out_dir / f"{row['subject']}.npz", **out)
        log.update(subject=row["subject"], seconds=round(time.time() - t0, 1))
        logs.append(log)
        print(f"[ok] {row['subject']}: {log['n_windows']} windows, "
              f"bads={log['bad_channels']}, ICA removed={log.get('ica_excluded', [])}, "
              f"MVAR order={log['mvar_order']} ({log['seconds']} s)")
    pd.DataFrame(logs).to_csv(out_dir / "preprocessing_log.csv", index=False)
    (out_dir / "config_used.json").write_text(json.dumps(cfg.__dict__, indent=2, default=str))


if __name__ == "__main__":
    main()

### `make_synthetic.py`: fake test EEG with a planted effect (testing only)

In [ ]:
%%writefile /content/pipeline/make_synthetic.py
"""Make a small synthetic EEG dataset to test the pipeline end-to-end.

This is NOT real data. It only checks that every step runs and that the models
can find a planted effect. Each subject gets 19-channel 10-20 EEG (1/f noise +
posterior alpha + eye blinks). In the "pain" groups, a theta (6 Hz) source at
Fz drives fronto-central channels with a short delay, so theta connectivity
(and directed PDC Fz -> others) increases with severity.

Labels written to the manifest:
  label_3class : 0 = control, 1 = mild, 2 = severe  (tests the SVM tree)
  label        : 0 = control, 1 = pain (mild or severe)
"""
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import mne

CH = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz",
      "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"]
DRIVEN = {"F3": 0.008, "F4": 0.012, "Cz": 0.016, "C3": 0.020, "C4": 0.020}


def pink(n, rng):
    f = np.fft.rfftfreq(n)
    spec = rng.standard_normal(len(f)) + 1j * rng.standard_normal(len(f))
    spec[1:] /= np.sqrt(f[1:])
    spec[0] = 0
    x = np.fft.irfft(spec, n)
    return x / x.std()


def narrowband(n, sfreq, lo, hi, rng):
    x = rng.standard_normal(n)
    x = mne.filter.filter_data(x[None], sfreq, lo, hi, verbose="error")[0]
    return x / x.std()


def make_subject(severity, sfreq, dur, rng):
    n = int(sfreq * dur)
    data = np.stack([pink(n, rng) for _ in CH]) * 8.0          # ~8 uV background
    alpha = narrowband(n, sfreq, 9, 11, rng)
    for i, ch in enumerate(CH):
        if ch[0] in "PO":
            data[i] += (6.0 - 1.5 * severity) * alpha * rng.uniform(0.8, 1.2)
        data[i] += 2.0 * narrowband(n, sfreq, 4, 8, rng)        # independent theta
    coupling = [0.0, 0.5, 1.0][severity]
    driver = narrowband(n, sfreq, 5, 7, rng) * 5.0
    data[CH.index("Fz")] += driver
    for ch, lag in DRIVEN.items():
        shift = int(lag * sfreq)
        data[CH.index(ch)] += coupling * np.roll(driver, shift)
    # eye blinks on frontal channels (gives ICA something to remove)
    for t in rng.uniform(1, dur - 1, size=int(dur / 4)):
        idx = int(t * sfreq)
        blink = 60 * np.hanning(int(0.3 * sfreq))
        for ch, g in (("Fp1", 1.0), ("Fp2", 1.0), ("F7", 0.4), ("F8", 0.4), ("F3", 0.3), ("F4", 0.3)):
            seg = data[CH.index(ch), idx:idx + len(blink)]
            seg += g * blink[:len(seg)]
    return data * 1e-6


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data/synthetic")
    ap.add_argument("--n-per-group", type=int, default=8)
    ap.add_argument("--duration", type=float, default=60.0)
    ap.add_argument("--sfreq", type=float, default=250.0)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    rng = np.random.default_rng(args.seed)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    info = mne.create_info(CH, args.sfreq, "eeg")
    from paineeg.preprocess import make_montage
    info.set_montage(make_montage("standard_1020"))
    rows = []
    k = 0
    for sev in (0, 1, 2):
        for _ in range(args.n_per_group):
            k += 1
            sid = f"sub-{k:02d}"
            raw = mne.io.RawArray(make_subject(sev, args.sfreq, args.duration, rng), info, verbose="error")
            fname = f"{sid}_eeg.fif"
            raw.save(out / fname, overwrite=True, verbose="error")
            rows.append(dict(subject=sid, path=fname, label=int(sev > 0), label_3class=sev))
    pd.DataFrame(rows).to_csv(out / "manifest.csv", index=False)
    print(f"Wrote {len(rows)} subjects to {out}/ and manifest.csv")


if __name__ == "__main__":
    main()

## 2. Code: Part 2 (Modares-Haghighi baseline)
### `paineeg/graph_metrics.py`: graph filtering + bctpy metrics

In [ ]:
%%writefile /content/pipeline/paineeg/graph_metrics.py
"""Graph filtering + graph metrics per window (Modares-Haghighi et al., 2021, sec. 3.3-3.4).

Shared by the SVM baseline and (later) the GAT, so both see the same graphs.

Graph filtering (applied to every window and band separately)
  cumulative thresholding : keep the strongest d of edges, d = 10 %, 20 %, ..., 90 %
  windowed thresholding   : keep edges whose strength rank lies in a band,
                            1-10 %, 11-20 %, ..., 61-70 % (strongest first)
Both are expressed as a rank interval (lo, hi]: cumulative d = (0, d].

Metrics (bctpy), on weights normalised to [0, 1] (divided by the window's max):
  degree       in + out degree (directed PDC) / degree (wPLI), per node
  betweenness  weighted, on lengths 1/w, per node
  clustering   clustering_coef_wd (directed) / clustering_coef_wu (wPLI), per node
  local_eff    weighted local efficiency (Wang et al., 2016), per node
  global_eff   weighted global efficiency, one value
Feature vector per window = [band1: 4 x C node metrics + 1 global, band2: ...].

Matrix convention (as in connectivity.py): W[source, target].
"""
import hashlib
from pathlib import Path

import numpy as np
import bct

NODE_METRICS = ("degree", "betweenness", "clustering", "local_eff")

# The paper's 16 graph versions: 9 cumulative + 7 windowed, as (name, lo, hi)
CUMULATIVE = [(f"cum{d}", 0.0, d / 100) for d in range(10, 100, 10)]
WINDOWED = [(f"win{lo + 1}-{lo + 10}", lo / 100, (lo + 10) / 100) for lo in range(0, 70, 10)]
GRAPH_VERSIONS = CUMULATIVE + WINDOWED


def version_for_density(density):
    """The cumulative version that keeps the strongest `density` of edges."""
    return (f"cum{int(round(density * 100))}", 0.0, float(density))


def filter_graph(W, lo, hi, directed):
    """Keep edges whose strength rank (strongest first) falls in (lo, hi] of all edges.

    Undirected graphs rank each pair once (upper triangle) and stay symmetric.
    """
    C = W.shape[0]
    if directed:
        idx = np.where(~np.eye(C, dtype=bool))
    else:
        idx = np.triu_indices(C, k=1)
    w = W[idx]
    order = np.argsort(-w, kind="stable")
    m = len(w)
    keep = order[int(round(lo * m)):int(round(hi * m))]
    out = np.zeros_like(W)
    rows, cols = idx[0][keep], idx[1][keep]
    out[rows, cols] = W[rows, cols]
    if not directed:
        out[cols, rows] = W[rows, cols]
    return out


def graph_features(W, directed):
    """Metrics of one filtered graph. Returns node metrics (4, C) and global efficiency."""
    mx = W.max()
    if mx <= 0:
        return np.zeros((len(NODE_METRICS), W.shape[0])), 0.0
    W = W / mx                                  # normalise weights to [0, 1]
    L = bct.weight_conversion(W, "lengths")     # length = 1 / weight
    if directed:
        _, _, deg = bct.degrees_dir(W)          # in + out
        clu = bct.clustering_coef_wd(W)
    else:
        deg = bct.degrees_und(W)
        clu = bct.clustering_coef_wu(W)
    btw = bct.betweenness_wei(L)
    eloc = bct.efficiency_wei(W, local=True)
    eglob = bct.efficiency_wei(W)
    node = np.vstack([deg, btw, np.ravel(clu), np.ravel(eloc)]).astype(float)
    return np.nan_to_num(node), float(np.nan_to_num(eglob))


def window_features(conn, directed, lo, hi):
    """Feature matrix for one subject. conn: (n_win, n_bands, C, C) -> (n_win, n_bands * (4C + 1))."""
    n_win, n_b, C, _ = conn.shape
    X = np.zeros((n_win, n_b, len(NODE_METRICS) * C + 1))
    for w in range(n_win):
        for b in range(n_b):
            node, glob = graph_features(filter_graph(conn[w, b], lo, hi, directed), directed)
            X[w, b, :-1] = node.ravel()
            X[w, b, -1] = glob
    return X.reshape(n_win, -1)


def feature_names(bands, ch_names):
    """Names matching window_features columns, e.g. 'theta_degree_Fz', 'alpha_global_eff'."""
    names = []
    for b in bands:
        names += [f"{b}_{m}_{ch}" for m in NODE_METRICS for ch in ch_names]
        names.append(f"{b}_global_eff")
    return names


def subject_features(subjects, lo, hi, cache_dir=None, n_jobs=1, tag=""):
    """window_features for every subject (from paineeg.data.load_subjects), in parallel.

    With `cache_dir`, results are stored per subject and graph version, keyed on the
    connectivity values, so sweeps and reruns reuse them.
    """
    from joblib import Parallel, delayed

    def one(s):
        path = None
        if cache_dir is not None:
            key = hashlib.sha1(np.ascontiguousarray(s["conn"]).tobytes()).hexdigest()[:12]
            path = Path(cache_dir) / f"{s['subject']}_{tag}_{lo:.2f}-{hi:.2f}_{key}.npy"
            if path.exists():
                return np.load(path)
        X = window_features(s["conn"], s["directed"], lo, hi)
        if path is not None:
            path.parent.mkdir(parents=True, exist_ok=True)
            np.save(path, X)
        return X

    return Parallel(n_jobs=n_jobs)(delayed(one)(s) for s in subjects)

### `02_baseline_modares.py`: SFFS + SVM, subject-wise CV, permutation test

In [ ]:
%%writefile /content/pipeline/02_baseline_modares.py
"""STEP 2 (model 1): Modares-Haghighi et al. (2021) baseline.

Graph metrics of the per-window connectivity graphs -> SFFS feature selection ->
RBF SVM (binary) or the paper's hierarchical tree of binary SVMs (multi-class).

Validation is always by SUBJECT: every window of one person is in the same fold.
Scaling, feature selection and C tuning are fitted inside the training folds only.

Usage
-----
python 02_baseline_modares.py --features features/mydata --conn wpli
python 02_baseline_modares.py --features features/mydata --conn pdc \
       --manifest data/mydata/manifest.csv --label-col label_3class --tree-order 0,2,1

Main options
  --density 0.9      cumulative threshold (keep strongest 90 % of edges; paper's best)
  --sweep            run all 16 graph versions (9 cumulative, 7 windowed) and use the best
  --loso             leave-one-subject-out instead of stratified 5-fold GroupKFold
  --paper-split      hold out ~25 % of subjects as a test set; 5-fold CV on the rest for tuning
  --n-perm 100       permutation test (labels shuffled at the subject level)
  --max-features 20  cap on SFFS features
  --fine-grid        C from 0.1 to 10 in 0.05 steps (paper) instead of a coarse grid
  --manifest CSV     take labels from this CSV (column --label-col, matched on 'subject')
  --noise-dirs A,B   also evaluate feature dirs built with 01_build_dataset.py --snr
"""
import argparse
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.feature_selection import f_classif
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix, accuracy_score,
                             roc_auc_score)
from sklearn.model_selection import (GridSearchCV, LeaveOneGroupOut, StratifiedGroupKFold,
                                     StratifiedShuffleSplit)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from paineeg.data import load_subjects
from paineeg import graph_metrics as gm

COARSE_C = [0.1, 0.3, 1.0, 3.0, 10.0]
FINE_C = list(np.round(np.arange(0.1, 10.0 + 1e-9, 0.05), 2))


# ------------------------------------------------------------------ data
def load_dataset(args, features_dir, version):
    """Window feature matrix, labels and subject ids for one graph version."""
    subjects = load_subjects(features_dir, conn=args.conn)
    if args.manifest:
        man = pd.read_csv(args.manifest).astype({"subject": str}).set_index("subject")
        for s in subjects:
            s["label"] = int(man.loc[s["subject"], args.label_col])
    elif args.label_col != "label":
        raise SystemExit(f"--label-col {args.label_col} needs --manifest (the .npz files "
                         "only store the label used when they were built)")
    name, lo, hi = version
    feats = gm.subject_features(subjects, lo, hi, cache_dir=Path(features_dir) / "graph_cache",
                                n_jobs=args.n_jobs, tag=args.conn)
    X = np.vstack(feats)
    groups = np.concatenate([[s["subject"]] * len(f) for s, f in zip(subjects, feats)])
    subj = pd.DataFrame(dict(subject=[s["subject"] for s in subjects],
                             label=[s["label"] for s in subjects]))
    names = gm.feature_names(subjects[0]["bands"], subjects[0]["ch_names"])
    return X, groups, subj, names


# ------------------------------------------------------------------ splits
def n_folds(labels, wanted):
    return max(2, min(wanted, np.bincount(labels).min() if len(labels) else 2))


def outer_splits(subj, args, seed):
    """Lists of (train_subjects, test_subjects)."""
    sid, lab = subj["subject"].to_numpy(), subj["label"].to_numpy()
    if args.loso:
        return [(np.delete(sid, i), sid[[i]]) for i in range(len(sid))]
    if args.paper_split:
        ss = StratifiedShuffleSplit(n_splits=1, test_size=args.test_size, random_state=seed)
        tr, te = next(ss.split(sid, lab))
        return [(sid[tr], sid[te])]
    k = n_folds(np.unique(lab, return_inverse=True)[1], args.folds)
    cv = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    return [(sid[tr], sid[te]) for tr, te in cv.split(sid, lab, groups=sid)]


def inner_cv(y_subj, args, seed):
    k = n_folds(y_subj, args.inner_folds)
    return StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)


def subsample(groups, per_subject):
    """Evenly spaced windows per subject (for SFFS and C tuning only)."""
    idx = []
    for g in np.unique(groups):
        w = np.where(groups == g)[0]
        if per_subject and len(w) > per_subject:
            w = w[np.linspace(0, len(w) - 1, per_subject).astype(int)]
        idx.append(w)
    return np.concatenate(idx)


# ------------------------------------------------------------------ one binary SVM node
def svm():
    return make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma="scale", C=1.0, class_weight="balanced"))


def fit_node(X, y, groups, args, seed, n_jobs):
    """SFFS + C tuning + final fit of one binary SVM, on training windows only."""
    feat = np.where(X.std(axis=0) > 1e-12)[0]               # drop zero-variance features
    if args.prefilter and len(feat) > args.prefilter:        # univariate pre-screen (train only)
        F, _ = f_classif(X[:, feat], y)
        feat = feat[np.argsort(-np.nan_to_num(F))[:args.prefilter]]
    sub = subsample(groups, args.sel_windows)
    Xs, ys, gs = X[np.ix_(sub, feat)], y[sub], groups[sub]
    y_subj = pd.Series(ys).groupby(gs).first().to_numpy()
    cv = inner_cv(y_subj, args, seed)
    k_max = min(args.max_features, len(feat))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sffs = SFS(svm(), k_features=(1, k_max), forward=True, floating=True,
                   scoring="balanced_accuracy", cv=cv, n_jobs=n_jobs, verbose=0)
        sffs.fit(Xs, ys, groups=gs)
    chosen = feat[list(sffs.k_feature_idx_)]
    grid = GridSearchCV(svm(), {"svc__C": FINE_C if args.fine_grid else COARSE_C},
                        scoring="balanced_accuracy", cv=cv, n_jobs=n_jobs)
    grid.fit(X[np.ix_(sub, chosen)], ys, groups=gs)
    C = grid.best_params_["svc__C"]
    model = svm().set_params(svc__C=C).fit(X[:, chosen], y)
    return dict(features=chosen, C=C, model=model, sffs_score=float(sffs.k_score_))


def fit_classifier(X, y, groups, classes, order, args, seed, n_jobs):
    """Binary: one SVM (positive = highest class). Multi-class: hierarchical tree that
    peels off one class per node, in `order` (Modares-Haghighi et al., 2021)."""
    if len(classes) == 2:
        pos = classes[-1]
        return [dict(peel=pos, **fit_node(X, (y == pos).astype(int), groups, args, seed, n_jobs))]
    nodes, remaining = [], list(order)
    for c in order[:-1]:
        m = np.isin(y, remaining)
        nodes.append(dict(peel=c, **fit_node(X[m], (y[m] == c).astype(int), groups[m],
                                             args, seed, n_jobs)))
        remaining.remove(c)
    return nodes


def predict(nodes, X, classes, order):
    """Window predictions and (binary only) decision scores for the positive class."""
    if len(classes) == 2:
        n = nodes[0]
        score = n["model"].decision_function(X[:, n["features"]])
        return np.where(score > 0, classes[-1], classes[0]), score
    pred = np.full(len(X), order[-1])
    undecided = np.ones(len(X), bool)
    for n in nodes:
        hit = undecided & (n["model"].decision_function(X[:, n["features"]]) > 0)
        pred[hit] = n["peel"]
        undecided &= ~hit
    return pred, None


# ------------------------------------------------------------------ metrics
def scores(y, pred, score, classes):
    out = dict(accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred))
    cm = confusion_matrix(y, pred, labels=classes)
    tp = np.diag(cm)
    fn = cm.sum(axis=1) - tp
    fp = cm.sum(axis=0) - tp
    tn = cm.sum() - tp - fn - fp
    with np.errstate(invalid="ignore", divide="ignore"):
        sens, spec = tp / (tp + fn), tn / (tn + fp)
    if len(classes) == 2:                       # positive class = classes[-1]
        out.update(sensitivity=sens[-1], specificity=sens[0])
        out["auc"] = roc_auc_score(y == classes[-1], score) if len(np.unique(y)) == 2 else np.nan
    else:
        out.update(sensitivity=np.nanmean(sens), specificity=np.nanmean(spec))
    return {k: float(v) for k, v in out.items()}


def subject_level(groups, y, pred, score, classes, rule):
    """Subject prediction = majority vote (or mean decision score) over its windows."""
    rows = []
    for g in pd.unique(groups):
        m = groups == g
        counts = np.array([(pred[m] == c).sum() for c in classes])
        row = dict(subject=g, true=int(y[m][0]), n_windows=int(m.sum()))
        if score is not None:
            row["mean_score"] = float(score[m].mean())
        if rule == "mean" and score is not None:
            row["pred"] = int(classes[-1] if row["mean_score"] > 0 else classes[0])
        elif score is not None and counts[0] == counts[-1]:   # binary tie -> mean score
            row["pred"] = int(classes[-1] if row["mean_score"] > 0 else classes[0])
        else:
            row["pred"] = int(classes[np.argmax(counts)])
        for c, n in zip(classes, counts):
            row[f"frac_{c}"] = n / m.sum()
        rows.append(row)
    return pd.DataFrame(rows)


# ------------------------------------------------------------------ one full evaluation
def evaluate(X, groups, subj, args, seed=0, n_jobs=1, keep_details=True):
    """Run the whole subject-wise CV. Returns a dict of results."""
    classes = [int(c) for c in sorted(subj["label"].unique())]
    order = [int(c) for c in args.tree_order.split(",")] if args.tree_order else classes
    if sorted(order) != classes:
        raise SystemExit(f"--tree-order {order} must list every class {classes}")
    lab = dict(zip(subj["subject"], subj["label"]))
    y = np.array([lab[g] for g in groups])
    folds, oof, subj_rows, selected = [], [], [], []
    for k, (tr_s, te_s) in enumerate(outer_splits(subj, args, seed)):
        tr, te = np.isin(groups, tr_s), np.isin(groups, te_s)
        nodes = fit_classifier(X[tr], y[tr], groups[tr], classes, order, args, seed, n_jobs)
        pred, score = predict(nodes, X[te], classes, order)
        sdf = subject_level(groups[te], y[te], pred, score, classes, args.subject_rule)
        sdf["fold"] = k
        w = scores(y[te], pred, score, classes)
        s = scores(sdf["true"].to_numpy(), sdf["pred"].to_numpy(),
                   sdf["mean_score"].to_numpy() if score is not None else None, classes)
        folds.append(dict(fold=k, n_test_subjects=len(te_s),
                          **{f"window_{m}": v for m, v in w.items()},
                          **{f"subject_{m}": v for m, v in s.items()},
                          C=";".join(str(n["C"]) for n in nodes)))
        oof.append(pd.DataFrame(dict(group=groups[te], y=y[te], pred=pred,
                                     score=score if score is not None else np.nan)))
        subj_rows.append(sdf)
        for i, n in enumerate(nodes):
            selected += [dict(fold=k, node=i, peel=n["peel"], feature=int(f)) for f in n["features"]]
    oof = pd.concat(oof)
    sdf = pd.concat(subj_rows, ignore_index=True)
    binary = len(classes) == 2
    pooled_w = scores(oof["y"].to_numpy(), oof["pred"].to_numpy(),
                      oof["score"].to_numpy() if binary else None, classes)
    pooled_s = scores(sdf["true"].to_numpy(), sdf["pred"].to_numpy(),
                      sdf["mean_score"].to_numpy() if binary else None, classes)
    res = dict(classes=classes, order=order, pooled_window=pooled_w, pooled_subject=pooled_s,
               per_fold=pd.DataFrame(folds))
    if keep_details:
        res.update(oof=oof, subjects=sdf, selected=pd.DataFrame(selected))
    return res


def permuted_score(X, groups, subj, args, i):
    rng = np.random.default_rng(10_000 + i)
    perm = subj.copy()
    perm["label"] = rng.permutation(perm["label"].to_numpy())
    r = evaluate(X, groups, perm, args, seed=args.seed, n_jobs=1, keep_details=False)
    return [r["pooled_subject"]["balanced_accuracy"], r["pooled_window"]["balanced_accuracy"],
            r["pooled_subject"].get("auc", np.nan)]


# ------------------------------------------------------------------ outputs
def summarise(per_fold):
    num = per_fold.drop(columns=["fold", "n_test_subjects", "C"])
    return {c: dict(mean=float(num[c].mean()), sd=float(num[c].std(ddof=1)) if len(num) > 1 else 0.0)
            for c in num.columns}


def plot_confusion(res, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    classes = res["classes"]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
    for ax, (title, yt, yp) in zip(axes, [
            ("Windows", res["oof"]["y"], res["oof"]["pred"]),
            ("Subjects", res["subjects"]["true"], res["subjects"]["pred"])]):
        cm = confusion_matrix(yt, yp, labels=classes)
        ax.imshow(cm, cmap="Blues")
        lo, hi = cm.min(), cm.max()
        for i in range(len(classes)):
            for j in range(len(classes)):
                # imshow scales colours from min to max, so pick the text colour the same way
                dark = hi > lo and (cm[i, j] - lo) / (hi - lo) > 0.5
                ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if dark else "black")
        ax.set_xticks(range(len(classes)), classes)
        ax.set_yticks(range(len(classes)), classes)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"{title} (pooled over folds)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def write_outputs(res, names, out, args, version, perm=None, extra=None):
    out.mkdir(parents=True, exist_ok=True)
    res["per_fold"].to_csv(out / "per_fold.csv", index=False)
    res["subjects"].to_csv(out / "subject_predictions.csv", index=False)
    sel = res["selected"]
    n_folds_run = res["per_fold"].shape[0]
    if len(sel):
        sel = (sel.assign(feature=[names[i] for i in sel["feature"]])
               .groupby(["node", "peel", "feature"]).size().rename("count").reset_index())
        sel["fraction_of_folds"] = sel["count"] / n_folds_run
        sel = sel.sort_values(["node", "count", "feature"], ascending=[True, False, True])
    sel.to_csv(out / "selected_features.csv", index=False)
    plot_confusion(res, out / "confusion_matrix.png")
    metrics = dict(conn=args.conn, label_col=args.label_col, classes=res["classes"],
                   tree_order=res["order"], graph_version=version[0],
                   rank_interval=[version[1], version[2]],
                   validation="loso" if args.loso else "paper-split" if args.paper_split
                   else f"stratified group {res['per_fold'].shape[0]}-fold",
                   pooled_window=res["pooled_window"], pooled_subject=res["pooled_subject"],
                   per_fold_mean_sd=summarise(res["per_fold"]))
    if perm is not None:
        metrics["permutation"] = perm
    if extra:
        metrics.update(extra)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    cfg = {k: v for k, v in vars(args).items()}
    cfg.update(graph_version=version[0], n_features_total=len(names))
    (out / "run_config.json").write_text(json.dumps(cfg, indent=2, default=str))


def fmt(d):
    return " ".join(f"{k}={v:.3f}" for k, v in d.items())


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--features", required=True, help="folder of .npz files from 01_build_dataset.py")
    ap.add_argument("--conn", choices=["pdc", "wpli"], default="pdc")
    ap.add_argument("--manifest", default=None, help="CSV with a 'subject' column and the label column")
    ap.add_argument("--label-col", default="label")
    ap.add_argument("--tree-order", default=None, help="class order for the multi-class tree, e.g. 0,2,1")
    ap.add_argument("--density", type=float, default=0.9)
    ap.add_argument("--sweep", action="store_true", help="evaluate all 16 graph versions")
    ap.add_argument("--loso", action="store_true")
    ap.add_argument("--paper-split", action="store_true")
    ap.add_argument("--test-size", type=float, default=0.25)
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--inner-folds", type=int, default=None,
                    help="CV folds inside training data for SFFS and C (default 3; 5 with --paper-split)")
    ap.add_argument("--n-perm", type=int, default=100)
    ap.add_argument("--max-features", type=int, default=20)
    ap.add_argument("--prefilter", type=int, default=30,
                    help="keep the N best features by ANOVA F (inside each training fold) before SFFS; 0 = off")
    ap.add_argument("--sel-windows", type=int, default=20,
                    help="windows per subject used by SFFS and C tuning (final model uses all); 0 = all")
    ap.add_argument("--fine-grid", action="store_true")
    ap.add_argument("--subject-rule", choices=["vote", "mean"], default="vote")
    ap.add_argument("--shuffle-labels", action="store_true",
                    help="leakage check: shuffle subject labels once before everything")
    ap.add_argument("--noise-dirs", default=None,
                    help="comma-separated feature dirs built with --snr; evaluated with the same settings")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--n-jobs", type=int, default=-1)
    ap.add_argument("--out", default=None, help="default results/baseline_<conn>/")
    args = ap.parse_args()
    if args.inner_folds is None:
        args.inner_folds = 5 if args.paper_split else 3
    if args.loso and args.paper_split:
        raise SystemExit("choose --loso or --paper-split, not both")
    out = Path(args.out or f"results/baseline_{args.conn}")
    t0 = time.time()

    def prepare(features_dir, version):
        X, groups, subj, names = load_dataset(args, features_dir, version)
        if args.shuffle_labels:
            subj["label"] = np.random.default_rng(args.seed).permutation(subj["label"].to_numpy())
        return X, groups, subj, names

    version = gm.version_for_density(args.density)
    sweep_rows = None
    if args.sweep:
        sweep_rows = []
        for v in gm.GRAPH_VERSIONS:
            X, groups, subj, names = prepare(args.features, v)
            r = evaluate(X, groups, subj, args, seed=args.seed, n_jobs=args.n_jobs, keep_details=False)
            sweep_rows.append(dict(version=v[0], lo=v[1], hi=v[2],
                                   **{f"window_{k}": x for k, x in r["pooled_window"].items()},
                                   **{f"subject_{k}": x for k, x in r["pooled_subject"].items()}))
            print(f"[sweep] {v[0]:>8}: subject bAcc={r['pooled_subject']['balanced_accuracy']:.3f} "
                  f"window bAcc={r['pooled_window']['balanced_accuracy']:.3f}")
        sw = pd.DataFrame(sweep_rows).sort_values(
            ["subject_balanced_accuracy", "window_balanced_accuracy"], ascending=False)
        out.mkdir(parents=True, exist_ok=True)
        sw.to_csv(out / "sweep.csv", index=False)
        best = sw.iloc[0]
        version = (best["version"], float(best["lo"]), float(best["hi"]))
        print(f"[sweep] best graph version: {version[0]} (chosen on the same CV -> optimistic)")

    X, groups, subj, names = prepare(args.features, version)
    print(f"{len(subj)} subjects, {len(X)} windows, {X.shape[1]} features, "
          f"classes {[int(c) for c in sorted(subj['label'].unique())]}, graph version {version[0]}")
    res = evaluate(X, groups, subj, args, seed=args.seed, n_jobs=args.n_jobs)
    print(f"[window ] {fmt(res['pooled_window'])}")
    print(f"[subject] {fmt(res['pooled_subject'])}")

    perm = None
    if args.n_perm > 0:
        null = Parallel(n_jobs=args.n_jobs, verbose=5)(          # prints progress
            delayed(permuted_score)(X, groups, subj, args, i) for i in range(args.n_perm))
        null = np.array(null)
        perm = dict(n_perm=args.n_perm, labels_shuffled_at="subject level")
        stats = [("subject_balanced_accuracy", res["pooled_subject"]["balanced_accuracy"]),
                 ("window_balanced_accuracy", res["pooled_window"]["balanced_accuracy"])]
        if "auc" in res["pooled_subject"]:          # binary: threshold-free subject statistic
            stats.append(("subject_auc", res["pooled_subject"]["auc"]))
        for i, (name, obs) in enumerate(stats):
            perm[name] = dict(observed=obs, null_mean=float(np.nanmean(null[:, i])),
                              p=float((1 + (null[:, i] >= obs).sum()) / (args.n_perm + 1)))
            print(f"[perm   ] {name}: observed {obs:.3f}, null mean {perm[name]['null_mean']:.3f}, "
                  f"p={perm[name]['p']:.3f}")

    extra = {}
    if args.noise_dirs:
        rows = [dict(features=args.features, snr="clean",
                     **{f"window_{k}": x for k, x in res["pooled_window"].items()},
                     **{f"subject_{k}": x for k, x in res["pooled_subject"].items()})]
        for d in args.noise_dirs.split(","):
            Xn, gn, sn, _ = prepare(d.strip(), version)
            r = evaluate(Xn, gn, sn, args, seed=args.seed, n_jobs=args.n_jobs, keep_details=False)
            rows.append(dict(features=d.strip(), snr=d.strip(),
                             **{f"window_{k}": x for k, x in r["pooled_window"].items()},
                             **{f"subject_{k}": x for k, x in r["pooled_subject"].items()}))
            print(f"[noise  ] {d}: subject bAcc={r['pooled_subject']['balanced_accuracy']:.3f} "
                  f"window bAcc={r['pooled_window']['balanced_accuracy']:.3f}")
        out.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(out / "noise_robustness.csv", index=False)

    extra["runtime_s"] = round(time.time() - t0, 1)
    if args.shuffle_labels:
        extra["labels_shuffled"] = True
    write_outputs(res, names, out, args, version, perm, extra)
    print(f"Saved results to {out}/ ({extra['runtime_s']} s)")


if __name__ == "__main__":
    main()

## 3. Connect Drive and choose settings

Data: `MyDrive/TRM/generic dataset` (the `.gdf` files). Features and results are saved to `MyDrive/TRM/`.
The mBrain recordings use **60 Hz mains**, so keep `LINE_FREQ = 60`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/pipeline
from pathlib import Path

dataset_dir = Path('/content/drive/MyDrive/TRM/generic dataset')
MANIFEST    = dataset_dir / 'manifest.csv'
FEATURES    = Path('/content/drive/MyDrive/TRM/features/generic_dataset')
RESULTS     = Path('/content/drive/MyDrive/TRM/results/generic_dataset')
CHANNELS    = 'Fp1,Fp2,F7,F3,Fz,F4,F8,T7,C3,Cz,C4,T8,P7,P3,Pz,P4,P8,O1,O2'
LINE_FREQ   = 60
N_PERM      = 100   # 0 = quick check (seconds instead of hours)

### Check the dataset

If `manifest.csv` is missing from the Drive folder, this cell writes it. It uses the same subjects and labels
as your local `generic_dataset/manifest.csv`. Then it checks that every listed `.gdf` file is present.

In [ ]:
import pandas as pd
MANIFEST_CSV = """subject,path,label,label_3class,pain_score
ID0,ID0.gdf,1,2,7.0
ID1,ID1.gdf,0,0,4.0
ID2,ID2.gdf,0,0,3.0
ID3,ID3.gdf,1,2,8.0
ID5,ID5.gdf,0,0,2.0
ID6,ID6.gdf,1,2,7.0
ID7,ID7.gdf,0,0,3.0
ID8,ID8.gdf,0,0,4.0
ID9,ID9.gdf,1,2,9.0
ID10,ID10.gdf,0,0,3.0
ID11,ID11.gdf,1,1,6.0
ID13,ID13.gdf,0,0,3.0
ID14,ID14.gdf,0,0,3.0
ID15,ID15.gdf,1,2,8.0
ID18,ID18.gdf,1,1,5.0
ID19,ID19.gdf,1,2,8.0
ID20,ID20.gdf,1,2,7.0
ID21,ID21.gdf,1,1,6.0
ID22,ID22.gdf,1,2,7.0
ID23,ID23.gdf,1,1,6.0
ID24,ID24.gdf,1,2,9.0
ID25,ID25.gdf,1,2,8.0
ID26,ID26.gdf,0,0,0.0
ID27,ID27.gdf,0,0,1.0
ID31,ID31.gdf,1,2,9.0
ID33,ID33.gdf,1,1,6.0
ID35,ID35.gdf,0,0,1.0
ID37,ID37.gdf,0,0,0.0
ID38,ID38.gdf,1,2,7.0
ID39,ID39.gdf,1,2,8.0
ID40,ID40.gdf,0,0,4.0
ID41,ID41.gdf,1,2,7.0
ID43,ID43.gdf,1,1,6.0
"""
assert dataset_dir.exists(), f'Not found: {dataset_dir} (check the folder name in Drive)'
if not MANIFEST.exists():
    MANIFEST.write_text(MANIFEST_CSV)
    print('wrote', MANIFEST)
man = pd.read_csv(MANIFEST)
missing = [p for p in man['path'] if not (dataset_dir / p).exists()]
print(f"{len(man)} subjects in manifest, {len(man) - len(missing)} .gdf files found")
print(man['label'].value_counts().rename('label').to_dict(), man['label_3class'].value_counts().rename('3class').to_dict())
assert not missing, f'missing files: {missing}'

### Optional: test on synthetic data first

In [ ]:
# !python make_synthetic.py --n-per-group 6
# !python 01_build_dataset.py --manifest data/synthetic/manifest.csv --out features/synthetic --conn both
# !python 02_baseline_modares.py --features features/synthetic --conn wpli --n-perm 0

## 4. Run Part 1: build the dataset (~8 s per subject)

Skip this if `FEATURES` already holds the `.npz` files, for example ones you uploaded from your laptop.

In [ ]:
!python 01_build_dataset.py --manifest "{MANIFEST}" --out "{FEATURES}" --conn both \
    --channels {CHANNELS} --line-freq {LINE_FREQ}

## 5. Run Part 2: baseline

With 100 permutations each run took 1.3–4.4 h on a 10-core laptop. Free Colab has 2 cores, so it
will take several times longer. Do one run with `N_PERM = 0` first. Each run's log goes to Drive
(`<run>.log`).

In [ ]:
def run(conn, name, extra=''):
    out = f'{RESULTS}/{name}'
    !mkdir -p "{out}"
    !python 02_baseline_modares.py --features "{FEATURES}" --conn {conn} --manifest "{MANIFEST}" \
        --n-perm {N_PERM} --n-jobs -1 --out "{out}" {extra} 2>&1 | grep -v -i warn | tee "{out}.log"

In [ ]:
run('wpli', 'wpli_binary')

In [ ]:
run('pdc', 'pdc_binary')

Optional: 3-class (the tree peels off class 0, then class 2; class 1 is what's left).

In [ ]:
run('wpli', 'wpli_3class', '--label-col label_3class --tree-order 0,2,1')
run('pdc',  'pdc_3class',  '--label-col label_3class --tree-order 0,2,1')

## 6. Results

In [ ]:
import json, glob
from IPython.display import Image, display
for m in sorted(glob.glob(f'{RESULTS}/*/metrics.json')):
    print('===', m.split('/')[-2])
    print(json.dumps(json.load(open(m)), indent=1)[:1500])
    display(Image(m.replace('metrics.json', 'confusion_matrix.png')))

---
# Part 3: Graph attention network (GAT)

This is a separate model from the baseline above. It reads the same features and uses the same graph filtering
(`--density 0.9`), the **same subject-wise folds** and the same metrics, so its results can be compared directly
with sections 5–6.

- **Graph per window:** nodes = the 19 channels. Node features = log band power (theta, alpha), z-scored on the
  training subjects. Edges = filtered connectivity, with one weight per band as edge features. PDC keeps its direction.
- **Model:** 2 × GATv2 layers (4 heads, 32 hidden units) → mean + max pooling → MLP. The loss is class-weighted.
- **Training:** in every outer fold, 20 % of the training subjects are held out for early stopping. The test
  subjects are only used for the final score.
- **Subject prediction:** majority vote of the subject's windows, the same rule as the baseline.

Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU). If the runtime restarted, run sections 0–3 again first,
because this part imports their code.

## 7. Install PyTorch Geometric

In [ ]:
!pip install -q torch_geometric
import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

## 8. Code: the GAT
### `paineeg/gat.py`: graph building + model

In [ ]:
%%writefile /content/pipeline/paineeg/gat.py
"""Graph attention network on the per-window connectivity graphs (model 2).

One graph per window: nodes = EEG channels, node features = log band power
(optionally plus the node's connectivity profile), edges = the connectivity graph
after the same filtering as the baseline (graph_metrics.filter_graph), with the
per-band weights as edge features.

Graphs are stored densely and turned into PyG edge lists one batch at a time on
the device, so 40k windows train quickly without building 40k Data objects.

Matrix convention (as in connectivity.py): W[source, target]; PyG messages flow
source -> target, so PDC keeps its direction. wPLI is symmetric (both directions).
"""
import numpy as np
import torch
from torch import nn
from torch_geometric.nn import GATv2Conv, global_max_pool, global_mean_pool

from . import graph_metrics as gm


# ------------------------------------------------------------------ graphs
def filtered_graphs(conn, directed, lo, hi):
    """(n_win, n_bands, C, C) -> same shape, filtered per window and band, scaled to [0, 1]
    by the window's strongest edge (as in graph_metrics.graph_features)."""
    out = np.zeros_like(conn)
    for w in range(conn.shape[0]):
        for b in range(conn.shape[1]):
            W = gm.filter_graph(conn[w, b], lo, hi, directed)
            mx = W.max()
            out[w, b] = W / mx if mx > 0 else W
    return out


def node_features(bandpower, graphs, kind):
    """bandpower (n_win, n_bands, C) -> (n_win, C, F).
    kind 'bp': log band power per band. 'bp+conn': plus each node's outgoing edge
    weights in every band (its connectivity profile)."""
    x = np.transpose(bandpower, (0, 2, 1))                      # (n_win, C, n_bands)
    if kind == "bp+conn":
        prof = np.transpose(graphs, (0, 2, 1, 3)).reshape(len(graphs), graphs.shape[2], -1)
        x = np.concatenate([x, prof], axis=2)
    return x.astype(np.float32)


def to_batch(x, A):
    """Dense batch -> PyG tensors. x: (B, C, F), A: (B, n_bands, C, C) on the device.
    An edge i -> j exists if it was kept in any band; its features are the band weights."""
    B, C, _ = x.shape
    b, i, j = torch.nonzero(A.amax(dim=1) > 0, as_tuple=True)
    edge_index = torch.stack([b * C + i, b * C + j])
    edge_attr = A[b, :, i, j]                                    # (E, n_bands)
    batch = torch.arange(B, device=x.device).repeat_interleave(C)
    return x.reshape(B * C, -1), edge_index, edge_attr, batch


# ------------------------------------------------------------------ model
class GAT(nn.Module):
    """GATv2 layers (edge-aware attention) -> mean + max pooling -> MLP classifier."""

    def __init__(self, in_dim, edge_dim, n_classes, hidden=32, heads=4, layers=2, dropout=0.3):
        super().__init__()
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        d = in_dim
        for _ in range(layers):
            self.convs.append(GATv2Conv(d, hidden, heads=heads, edge_dim=edge_dim,
                                        dropout=dropout, add_self_loops=True, fill_value="mean"))
            self.norms.append(nn.LayerNorm(hidden * heads))
            d = hidden * heads
        self.dropout = dropout
        self.head = nn.Sequential(nn.Linear(2 * d, hidden), nn.ELU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, n_classes))

    def forward(self, x, edge_index, edge_attr, batch, return_attention=False):
        att = None
        for k, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            x = nn.functional.dropout(x, self.dropout, self.training)
            if return_attention and k == len(self.convs) - 1:
                x, (ei, alpha) = conv(x, edge_index, edge_attr, return_attention_weights=True)
                att = (ei, alpha.mean(dim=1))                    # average over heads
            else:
                x = conv(x, edge_index, edge_attr)
            x = nn.functional.elu(norm(x))
        g = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        logits = self.head(g)
        return (logits, att) if return_attention else logits

### `03_gat.py`: training with subject-wise CV, results, attention

In [ ]:
%%writefile /content/pipeline/03_gat.py
"""STEP 3 (model 2): graph attention network (GAT) on the per-window connectivity graphs.

Reads the same .npz files as the baseline, applies the same graph filtering
(--density), and uses the SAME subject-wise outer folds and metrics
(02_baseline_modares.py: outer_splits, scores, subject_level), so the two models
are compared on identical splits.

Per window: nodes = channels, node features = log band power (z-scored on the
training subjects), edges = filtered connectivity with one weight per band.
Inside each outer fold, a share of the training subjects (--val-frac) is held out
for early stopping; test subjects are only used to score the final model.

Usage
-----
python 03_gat.py --features features/mydata --conn wpli
python 03_gat.py --features features/mydata --conn pdc \
       --manifest data/mydata/manifest.csv --label-col label_3class

Main options
  --density 0.9          keep the strongest 90 % of edges (same as the baseline)
  --node-features bp     'bp' = band power; 'bp+conn' adds each node's edge weights
  --hidden 32 --heads 4 --layers 2 --dropout 0.3
  --epochs 100 --patience 15 --batch 256 --lr 1e-3 --weight-decay 1e-4
  --ensemble 1           models per fold with different seeds, logits averaged
  --n-perm 0             permutation test (subject labels shuffled, whole CV rerun)
  --loso / --paper-split as in the baseline
"""
import argparse
import importlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit

from paineeg.data import load_subjects
from paineeg import graph_metrics as gm
from paineeg import gat as G

base = importlib.import_module("02_baseline_modares")      # shared splits and metrics


# ------------------------------------------------------------------ data
def load_dataset(args):
    """Dense graphs, node features, labels and subject ids for every window."""
    subjects = load_subjects(args.features, conn=args.conn)
    if args.manifest:
        man = pd.read_csv(args.manifest).astype({"subject": str}).set_index("subject")
        for s in subjects:
            s["label"] = int(man.loc[s["subject"], args.label_col])
    elif args.label_col != "label":
        raise SystemExit(f"--label-col {args.label_col} needs --manifest")
    _, lo, hi = gm.version_for_density(args.density)
    A, X, groups = [], [], []
    for s in subjects:
        g = G.filtered_graphs(s["conn"], s["directed"], lo, hi)
        A.append(g)
        X.append(G.node_features(s["bandpower"], g, args.node_features))
        groups += [s["subject"]] * len(g)
    subj = pd.DataFrame(dict(subject=[s["subject"] for s in subjects],
                             label=[s["label"] for s in subjects]))
    return (np.concatenate(X), np.concatenate(A).astype(np.float32), np.array(groups), subj,
            subjects[0]["ch_names"], subjects[0]["bands"])


# ------------------------------------------------------------------ training
def batches(idx, size, rng=None):
    if rng is not None:
        idx = rng.permutation(idx)
    for k in range(0, len(idx), size):
        yield idx[k:k + size]


def run_model(model, X, A, idx, args, return_attention=False):
    """Logits (and last-layer attention) for windows idx, in eval mode."""
    model.eval()
    out, att = [], []
    with torch.no_grad():
        for b in batches(idx, args.batch * 4):
            x, ei, ea, bt = G.to_batch(X[b], A[b])
            r = model(x, ei, ea, bt, return_attention=return_attention)
            if return_attention:
                r, a = r
                att.append((b, a[0].cpu().numpy(), a[1].cpu().numpy()))
            out.append(r.float().cpu())
    logits = torch.cat(out).numpy()
    return (logits, att) if return_attention else logits


def train_one(X, A, yi, tr, va, n_classes, args, seed, device):
    """Train on windows tr, early-stop on validation loss over windows va."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = G.GAT(X.shape[2], A.shape[1], n_classes, args.hidden, args.heads,
                  args.layers, args.dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    counts = np.bincount(yi[tr], minlength=n_classes)
    w = torch.tensor(len(tr) / (n_classes * np.maximum(counts, 1)), dtype=torch.float32, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)           # balanced, as class_weight in the SVM
    y_dev = torch.as_tensor(yi, device=device)
    best, best_loss, wait, log = None, np.inf, 0, []
    for ep in range(args.epochs):
        model.train()
        tot = 0.0
        for b in batches(tr, args.batch, rng):
            x, ei, ea, bt = G.to_batch(X[b], A[b])
            loss = loss_fn(model(x, ei, ea, bt), y_dev[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)
        lv = run_model(model, X, A, va, args)
        v_loss = float(loss_fn(torch.as_tensor(lv, device=device), y_dev[va]))
        v_bacc = balanced_accuracy_score(yi[va], lv.argmax(1))
        log.append(dict(epoch=ep, train_loss=tot / len(tr), val_loss=v_loss, val_window_bacc=v_bacc))
        if v_loss < best_loss - 1e-4:
            best_loss, wait = v_loss, 0
            best = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= args.patience:
                break
    model.load_state_dict(best)
    return model, pd.DataFrame(log)


def evaluate(X_np, A_np, groups, subj, args, device, seed, keep_details=True, verbose=True):
    """Whole subject-wise CV (same outer folds as the baseline)."""
    classes = [int(c) for c in sorted(subj["label"].unique())]
    lab = dict(zip(subj["subject"], subj["label"]))
    y = np.array([lab[g] for g in groups])
    yi = np.searchsorted(classes, y)
    binary = len(classes) == 2
    C = X_np.shape[1]
    A = torch.as_tensor(A_np, device=device)
    folds, oof, subj_rows, logs = [], [], [], []
    att_sum = np.zeros((len(classes), C, C))
    att_n = np.zeros((len(classes), C, C))
    for k, (tr_s, te_s) in enumerate(base.outer_splits(subj, args, seed)):
        t0 = time.time()
        # validation subjects for early stopping, taken from the training subjects
        tr_lab = subj.set_index("subject").loc[tr_s, "label"].to_numpy()
        n_val = max(len(classes), int(round(args.val_frac * len(tr_s))))
        ss = StratifiedShuffleSplit(n_splits=1, test_size=n_val, random_state=seed + k)
        fit_i, val_i = next(ss.split(tr_s, tr_lab))
        tr = np.where(np.isin(groups, tr_s[fit_i]))[0]
        va = np.where(np.isin(groups, tr_s[val_i]))[0]
        te = np.where(np.isin(groups, te_s))[0]
        # z-score node features with the fitting windows only
        mu, sd = X_np[tr].mean(axis=0), X_np[tr].std(axis=0) + 1e-6
        X = torch.as_tensor((X_np - mu) / sd, device=device)
        logits, epochs = 0, []
        for e in range(args.ensemble):
            model, log = train_one(X, A, yi, tr, va, len(classes), args, seed + 100 * e + k, device)
            lt, att = run_model(model, X, A, te, args, return_attention=True)
            logits = logits + lt / args.ensemble
            epochs.append(len(log))
            logs.append(log.assign(fold=k, member=e))
            for b, ei, alpha in att:                          # last-layer attention per class
                win = b[ei[0] // C]
                ci = yi[win]
                np.add.at(att_sum, (ci, ei[0] % C, ei[1] % C), alpha)
                np.add.at(att_n, (ci, ei[0] % C, ei[1] % C), 1)
        pred = np.array(classes)[logits.argmax(1)]
        score = logits[:, -1] - logits[:, 0] if binary else None
        sdf = base.subject_level(groups[te], y[te], pred, score, classes, args.subject_rule)
        sdf["fold"] = k
        w = base.scores(y[te], pred, score, classes)
        s = base.scores(sdf["true"].to_numpy(), sdf["pred"].to_numpy(),
                        sdf["mean_score"].to_numpy() if binary else None, classes)
        folds.append(dict(fold=k, n_test_subjects=len(te_s),
                          **{f"window_{m}": v for m, v in w.items()},
                          **{f"subject_{m}": v for m, v in s.items()},
                          epochs=";".join(map(str, epochs))))
        oof.append(pd.DataFrame(dict(group=groups[te], y=y[te], pred=pred,
                                     score=score if binary else np.nan)))
        subj_rows.append(sdf)
        if verbose:
            print(f"  fold {k}: subject bAcc={s['balanced_accuracy']:.3f} window bAcc="
                  f"{w['balanced_accuracy']:.3f} epochs={epochs} ({time.time() - t0:.0f} s)", flush=True)
    oof = pd.concat(oof)
    sdf = pd.concat(subj_rows, ignore_index=True)
    res = dict(classes=classes, order=classes,
               pooled_window=base.scores(oof["y"].to_numpy(), oof["pred"].to_numpy(),
                                         oof["score"].to_numpy() if binary else None, classes),
               pooled_subject=base.scores(sdf["true"].to_numpy(), sdf["pred"].to_numpy(),
                                          sdf["mean_score"].to_numpy() if binary else None, classes),
               per_fold=pd.DataFrame(folds))
    if keep_details:
        with np.errstate(invalid="ignore"):
            res.update(oof=oof, subjects=sdf, train_log=pd.concat(logs, ignore_index=True),
                       attention=att_sum / att_n)
    return res


# ------------------------------------------------------------------ outputs
def attention_table(att, classes, ch_names):
    """Mean last-layer attention per edge (source -> target) and class, strongest first."""
    rows = []
    for c, M in zip(classes, att):
        for i, src in enumerate(ch_names):
            for j, dst in enumerate(ch_names):
                if np.isfinite(M[i, j]):
                    rows.append(dict(true_class=c, source=src, target=dst, attention=M[i, j]))
    return pd.DataFrame(rows).sort_values(["true_class", "attention"], ascending=[True, False])


def write_outputs(res, args, out, ch_names, perm, extra):
    out.mkdir(parents=True, exist_ok=True)
    res["per_fold"].to_csv(out / "per_fold.csv", index=False)
    res["subjects"].to_csv(out / "subject_predictions.csv", index=False)
    res["train_log"].to_csv(out / "training_log.csv", index=False)
    attention_table(res["attention"], res["classes"], ch_names).to_csv(out / "attention_edges.csv", index=False)
    base.plot_confusion(res, out / "confusion_matrix.png")
    num = res["per_fold"].drop(columns=["fold", "n_test_subjects", "epochs"])
    metrics = dict(model="GAT", conn=args.conn, label_col=args.label_col, classes=res["classes"],
                   graph_version=gm.version_for_density(args.density)[0],
                   validation="loso" if args.loso else "paper-split" if args.paper_split
                   else f"stratified group {len(num)}-fold",
                   pooled_window=res["pooled_window"], pooled_subject=res["pooled_subject"],
                   per_fold_mean_sd={c: dict(mean=float(num[c].mean()),
                                             sd=float(num[c].std(ddof=1)) if len(num) > 1 else 0.0)
                                     for c in num.columns})
    if perm is not None:
        metrics["permutation"] = perm
    metrics.update(extra)
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (out / "run_config.json").write_text(json.dumps(vars(args), indent=2, default=str))


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--features", required=True, help="folder of .npz files from 01_build_dataset.py")
    ap.add_argument("--conn", choices=["pdc", "wpli"], default="wpli")
    ap.add_argument("--manifest", default=None, help="CSV with a 'subject' column and the label column")
    ap.add_argument("--label-col", default="label")
    ap.add_argument("--density", type=float, default=0.9)
    ap.add_argument("--node-features", choices=["bp", "bp+conn"], default="bp")
    ap.add_argument("--hidden", type=int, default=32)
    ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--layers", type=int, default=2)
    ap.add_argument("--dropout", type=float, default=0.3)
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--patience", type=int, default=15)
    ap.add_argument("--batch", type=int, default=256)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--weight-decay", type=float, default=1e-4)
    ap.add_argument("--val-frac", type=float, default=0.2, help="share of training subjects for early stopping")
    ap.add_argument("--ensemble", type=int, default=1)
    ap.add_argument("--loso", action="store_true")
    ap.add_argument("--paper-split", action="store_true")
    ap.add_argument("--test-size", type=float, default=0.25)
    ap.add_argument("--folds", type=int, default=5)
    ap.add_argument("--n-perm", type=int, default=0)
    ap.add_argument("--subject-rule", choices=["vote", "mean"], default="vote")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default=None, help="cuda / cpu / mps (default: cuda if available)")
    ap.add_argument("--out", default=None, help="default results/gat_<conn>/")
    args = ap.parse_args()
    if args.loso and args.paper_split:
        raise SystemExit("choose --loso or --paper-split, not both")
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    out = Path(args.out or f"results/gat_{args.conn}")
    t0 = time.time()

    X, A, groups, subj, ch_names, bands = load_dataset(args)
    print(f"{len(subj)} subjects, {len(X)} windows, {X.shape[1]} nodes x {X.shape[2]} features, "
          f"edge features {bands}, classes {sorted(subj['label'].unique().tolist())}, device {device}")
    res = evaluate(X, A, groups, subj, args, device, args.seed)
    print(f"[window ] {base.fmt(res['pooled_window'])}")
    print(f"[subject] {base.fmt(res['pooled_subject'])}")

    perm = None
    if args.n_perm > 0:
        null = []
        for i in range(args.n_perm):
            p = subj.copy()
            p["label"] = np.random.default_rng(10_000 + i).permutation(p["label"].to_numpy())
            r = evaluate(X, A, groups, p, args, device, args.seed, keep_details=False, verbose=False)
            null.append([r["pooled_subject"]["balanced_accuracy"], r["pooled_window"]["balanced_accuracy"],
                         r["pooled_subject"].get("auc", np.nan)])
            print(f"  perm {i + 1}/{args.n_perm}: subject bAcc={null[-1][0]:.3f}", flush=True)
        null = np.array(null)
        perm = dict(n_perm=args.n_perm, labels_shuffled_at="subject level")
        stats = [("subject_balanced_accuracy", res["pooled_subject"]["balanced_accuracy"]),
                 ("window_balanced_accuracy", res["pooled_window"]["balanced_accuracy"])]
        if "auc" in res["pooled_subject"]:
            stats.append(("subject_auc", res["pooled_subject"]["auc"]))
        for i, (name, obs) in enumerate(stats):
            perm[name] = dict(observed=obs, null_mean=float(np.nanmean(null[:, i])),
                              p=float((1 + (null[:, i] >= obs).sum()) / (args.n_perm + 1)))
            print(f"[perm   ] {name}: observed {obs:.3f}, null mean {perm[name]['null_mean']:.3f}, "
                  f"p={perm[name]['p']:.3f}")

    extra = dict(runtime_s=round(time.time() - t0, 1), device=str(device))
    write_outputs(res, args, out, ch_names, perm, extra)
    print(f"Saved results to {out}/ ({extra['runtime_s']} s)")


if __name__ == "__main__":
    main()

## 9. Run the GAT

`GAT_PERM` = permutation test. It retrains the whole CV for each shuffle, so leave it at 0 until the normal runs work.
Logs and results are saved to `RESULTS/gat_<run>/`.

In [ ]:
GAT_PERM = 0

def run_gat(conn, name, extra=''):
    out = f'{RESULTS}/{name}'
    !mkdir -p "{out}"
    !python 03_gat.py --features "{FEATURES}" --conn {conn} --manifest "{MANIFEST}" \
        --n-perm {GAT_PERM} --out "{out}" {extra} 2>&1 | grep -v -i warn | tee "{out}.log"

In [ ]:
run_gat('wpli', 'gat_wpli_binary')

In [ ]:
run_gat('pdc', 'gat_pdc_binary')

Optional: 3-class (softmax over the three classes; there's no tree).

In [ ]:
run_gat('wpli', 'gat_wpli_3class', '--label-col label_3class')
run_gat('pdc',  'gat_pdc_3class',  '--label-col label_3class')

Other settings you can add as `extra`: `--node-features bp+conn` (adds each node's connectivity profile),
`--ensemble 5` (5 models per fold, averaged), `--layers 3`, `--hidden 64`, `--dropout 0.5`, `--loso`.

## 10. GAT results and comparison with the baseline

In [ ]:
import json, glob
import pandas as pd
from IPython.display import Image, display

rows = []
for m in sorted(glob.glob(f'{RESULTS}/*/metrics.json')):
    d = json.load(open(m)); name = m.split('/')[-2]
    rows.append(dict(run=name, model=d.get('model', 'SVM baseline'), conn=d['conn'], labels=d['label_col'],
                     subject_bacc=d['pooled_subject']['balanced_accuracy'],
                     subject_auc=d['pooled_subject'].get('auc'),
                     window_bacc=d['pooled_window']['balanced_accuracy'],
                     perm_p=d.get('permutation', {}).get('subject_balanced_accuracy', {}).get('p')))
display(pd.DataFrame(rows).sort_values(['labels', 'conn', 'model']).round(3))

for m in sorted(glob.glob(f'{RESULTS}/gat_*/metrics.json')):
    run = m.rsplit('/', 2)[-2]
    print('===', run)
    display(Image(m.replace('metrics.json', 'confusion_matrix.png')))
    att = pd.read_csv(m.replace('metrics.json', 'attention_edges.csv'))
    print('Edges the GAT attends to most, per true class:')
    display(att[att.source != att.target].groupby('true_class').head(5))